# 04. Agentic Loop & `load_model` — the canonical entry point

`load_model()` is the front door to ArcLLM. You hand it a provider name and
get back a fully-wired adapter — circuit breakers, queues, telemetry, audit
sinks, retries, fallbacks, rate limits, OTel exporters, classification
routers — composed in a deterministic order. Every higher-level package
(`arcrun`, `arcagent`, `arcteam`, `arcgateway`) calls into this surface.

This notebook walks the API end-to-end. It is **mock-first**: every cell
runs without an API key by stubbing the underlying adapter. A short
`(live)` section at the end exercises a real provider when
`ANTHROPIC_API_KEY` is set.

You will learn:

- How an "agentic loop" is built in arcllm — adapter + module stack
- Every kwarg `load_model()` accepts, with a worked example for each
- The default module stack order (Otel → Queue → Telemetry → Audit →
  Guardrails → Injection → Security → CircuitBreaker → Retry → Fallback →
  RateLimit) and why the order matters
- How to feed a `trace_store`, an `on_event` callback, an `agent_label`,
  and per-call classification routing
- How to inspect the actual `_inner` chain on a constructed model so you
  can debug stack composition by eye

If you have not read `03-anthropic-adapter.ipynb` (the adapter contract)
yet, start there — this notebook treats adapters as a black box and
focuses on what wraps them.


## 1. Setup

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

In [2]:
# (live) optional — set this to run real-API sections; mock cells run regardless
HAS_LIVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"Live API key: {'present' if HAS_LIVE_KEY else 'missing — live cells will skip'}")

Live API key: missing — live cells will skip


Pull in the public surface from `arcllm`. We'll lean on `load_model`,
`clear_cache`, the `LLMProvider` ABC, and the data types we'll feed through
`invoke()`. A few module classes (`TelemetryModule`, `OtelModule`, etc.) are
imported for stack-walking demos.

In [3]:
from unittest.mock import AsyncMock, MagicMock, patch

import arcllm
from arcllm import (
    AnthropicAdapter,
    AuditModule,
    BaseModule,
    LLMResponse,
    Message,
    OtelModule,
    QueueModule,
    RateLimitModule,
    RetryModule,
    SecurityModule,
    TelemetryModule,
    TextBlock,
    Tool,
    ToolCall,
    ToolResultBlock,
    ToolUseBlock,
    Usage,
    clear_cache,
    load_model,
    load_provider_config,
)
from arcllm.modules.circuit_breaker import CircuitBreakerModule
from arcllm.modules.routing import RoutingModule
from arcllm.types import LLMProvider
from arcllm.exceptions import QueueFullError, ArcLLMConfigError

print("arcllm version:", arcllm.__version__)

arcllm version: 0.7.0


Reset the registry caches and clear any module-level rate-limit / OTel
state so this notebook starts from a known baseline. `clear_cache()` is
the standard test-isolation hook — call it whenever you want a fresh
slate.

In [4]:
clear_cache()
print("registry caches cleared")

registry caches cleared


Set a fake API key for the rest of the notebook. The mocks short-circuit
real HTTP, but the adapter's constructor still validates that *some*
key is reachable.

In [5]:
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key-not-real")
os.environ.setdefault("OPENAI_API_KEY", "test-key-not-real")
print("API keys set (test values).")

API keys set (test values).


## 2. The agentic loop in arcllm

An *agentic loop* is the cycle:

```
LLM → tool_use → execute tool → tool_result → LLM → (repeat or end_turn)
```

Most loop logic — turn counting, tool dispatch, prompt assembly — lives
in `arcrun`. arcllm's job is narrower: **provide a single async
callable** that, given messages and tools, returns an `LLMResponse`. That
callable can be a bare adapter, or it can be wrapped in a stack of
modules that add resilience, telemetry, and policy.

`load_model()` builds that callable. Reading the rest of this notebook
in one sentence: every kwarg you pass either turns on a wrapper or
threads a callback into a wrapper.

Before we touch `load_model`, a quick reminder of what an adapter looks
like *naked*. We'll mock httpx so it runs offline.

In [6]:
def fake_post_response(content_text: str = "Hi from mock!", usage_in=10, usage_out=5):
    """Build a canned Anthropic Messages API JSON envelope."""
    return {
        "id": "msg_mock",
        "type": "message",
        "role": "assistant",
        "model": "claude-sonnet-4-6",
        "content": [{"type": "text", "text": content_text}],
        "stop_reason": "end_turn",
        "usage": {"input_tokens": usage_in, "output_tokens": usage_out},
    }


class MockHttpxResponse:
    """Mimic httpx.Response just enough for AnthropicAdapter to parse."""

    def __init__(self, json_payload):
        self._json = json_payload
        self.status_code = 200

    def json(self):
        return self._json

    def raise_for_status(self):
        return None


print("mock helpers ready")

mock helpers ready


With those helpers, we can construct an adapter and call `invoke()`
without ever hitting the network.

In [7]:
cfg = load_provider_config("anthropic")
adapter = AnthropicAdapter(cfg, "claude-sonnet-4-6")

with patch.object(
    adapter._client,
    "post",
    new=AsyncMock(return_value=MockHttpxResponse(fake_post_response("2 + 2 = 4."))),
):
    resp = await adapter.invoke([Message(role="user", content="2+2?")])

print("type:", type(resp).__name__)
print("content:", resp.content)
print("stop_reason:", resp.stop_reason)
print("usage:", resp.usage)
await adapter.close()

type: LLMResponse
content: 2 + 2 = 4.
stop_reason: end_turn
usage: input_tokens=10 output_tokens=5 total_tokens=15 cache_read_tokens=None cache_write_tokens=None reasoning_tokens=None


Good — that's the bare adapter. The rest of this notebook is about
**what wraps it** when you call `load_model()` instead of constructing
the adapter directly.

## 3. Building a loop manually — pre-`load_model` demo

Before we use `load_model`, let's stack one or two modules around an
adapter by hand. This makes the abstraction concrete: a module is just a
class that takes `(config, inner)` and exposes the same `invoke()`
contract as the adapter underneath.

We'll use a `MagicMock(spec=LLMProvider)` as the inner so we don't need
real API calls.

In [8]:
def make_mock_inner(
    *,
    name: str = "anthropic",
    model: str = "claude-sonnet-4-6",
    response: LLMResponse | None = None,
    raises: Exception | None = None,
) -> LLMProvider:
    """Reusable mock inner provider for stack composition demos."""
    inner = MagicMock(spec=LLMProvider)
    inner.name = name
    inner.model_name = model
    inner.validate_config.return_value = True
    if raises is not None:
        inner.invoke = AsyncMock(side_effect=raises)
    else:
        inner.invoke = AsyncMock(
            return_value=response
            or LLMResponse(
                content="hello from mock",
                usage=Usage(input_tokens=12, output_tokens=4, total_tokens=16),
                model=model,
                stop_reason="end_turn",
            )
        )
    return inner


inner = make_mock_inner()
print("mock inner ready —", inner.name, inner.model_name)

mock inner ready — anthropic claude-sonnet-4-6


Wrap it in `AuditModule`, then call invoke. The audit module's only
job is to log metadata around the call; the response object passes
through unchanged.

In [9]:
audit_only = AuditModule({}, make_mock_inner())
resp = await audit_only.invoke([Message(role="user", content="hi")])
print("content:", resp.content, "| usage.total:", resp.usage.total_tokens)

content: hello from mock | usage.total: 16


Now stack two modules — `RetryModule` outside, `AuditModule` inside.
The retry module catches transient errors from its inner. We'll feed it
a flaky inner that fails twice and then succeeds.

In [10]:
attempts = {"n": 0}


def flaky_invoke(*_args, **_kwargs):
    attempts["n"] += 1
    if attempts["n"] < 3:
        from arcllm.exceptions import ArcLLMAPIError

        raise ArcLLMAPIError(503, "transient 503", "anthropic")
    return LLMResponse(
        content="recovered",
        usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    )


flaky_inner = MagicMock(spec=LLMProvider)
flaky_inner.name = "anthropic"
flaky_inner.model_name = "claude-sonnet-4-6"
flaky_inner.invoke = AsyncMock(side_effect=flaky_invoke)

audit_inner = AuditModule({}, flaky_inner)
retry_outer = RetryModule({"max_retries": 4, "backoff_base_seconds": 0.01}, audit_inner)

resp = await retry_outer.invoke([Message(role="user", content="hi")])
print(f"after {attempts['n']} attempts:", resp.content)

Retry attempt 1/4 after 0.01s: anthropic API error (HTTP 503): transient 503


Retry attempt 2/4 after 0.03s: anthropic API error (HTTP 503): transient 503


after 3 attempts: recovered


That's the entire mental model. `load_model()` does this same
composition for you — but in a fixed, audited order.

## 4. `load_model` basics — provider, model, and what you get for free

The simplest call is two arguments: provider name + model name. Provider
must match a TOML file under `arcllm/providers/` and a module under
`arcllm/adapters/`; model is optional and falls back to the provider's
`default_model`.

In [11]:
clear_cache()
m = load_model("anthropic")
print("model_name:", m.model_name)
print("type:", type(m).__name__)

model_name: claude-sonnet-4-6
type: QueueModule


Notice the type is **not** `AnthropicAdapter` directly — it's whatever
sits at the top of the default stack (typically `OtelModule` or
`QueueModule` or `TelemetryModule`, depending on which modules are
enabled in your `config.toml`). The full chain is composed for you.

You can confirm which modules are active by walking the `_inner`
attribute. Every `BaseModule` exposes `_inner`; the adapter at the
bottom does not.

In [12]:
def stack_layers(provider) -> list[str]:
    """Return [outermost, ..., innermost] class names for a load_model() result."""
    layers = []
    cur = provider
    while True:
        layers.append(type(cur).__name__)
        inner = getattr(cur, "_inner", None)
        if inner is None:
            break
        cur = inner
    return layers


print("default stack:", " -> ".join(stack_layers(m)))

default stack: QueueModule -> TelemetryModule -> AuditModule -> RetryModule -> FallbackModule -> RateLimitModule -> AnthropicAdapter


Disable everything (`telemetry=False, retry=False, queue=False,
security=False`, etc.) to see a bare adapter come back.

In [13]:
bare = load_model(
    "anthropic",
    telemetry=False,
    retry=False,
    queue=False,
    security=False,
    audit=False,
    rate_limit=False,
    fallback=False,
    circuit_breaker=False,
    otel=False,
)
print("bare stack:", " -> ".join(stack_layers(bare)))
print("isinstance(AnthropicAdapter):", isinstance(bare, AnthropicAdapter))

bare stack: AnthropicAdapter
isinstance(AnthropicAdapter): True


And turn just *one* module on with a config dict to see the shape of
the API.

In [14]:
queued = load_model(
    "anthropic",
    telemetry=False,
    retry=False,
    security=False,
    audit=False,
    rate_limit=False,
    fallback=False,
    circuit_breaker=False,
    otel=False,
    queue={"max_concurrent": 4, "call_timeout": 10.0},
)
print("queue-only stack:", " -> ".join(stack_layers(queued)))
print("queue capacity:", queued._max_concurrent, "timeout:", queued._call_timeout)

queue-only stack: QueueModule -> AnthropicAdapter
queue capacity: 4 timeout: 10.0


**Three-state semantics for every module kwarg:**

| Value      | Meaning                                                   |
|------------|------------------------------------------------------------|
| `None`     | Use config.toml `enabled` flag (this is the default)      |
| `True`     | Force-enable with config.toml settings                    |
| `False`    | Force-disable, even if config.toml enables it              |
| `dict`     | Force-enable; merge dict over config.toml settings        |

This is the lever you pull to override `arcllm.toml` without touching
the file — useful for tests, ephemeral agents, or per-call overrides.

## 5. The `on_event` callback

`on_event` is a synchronous callable that fires after every `invoke()`
with a `TraceRecord` describing what happened: tokens, cost, latency,
phase sub-timings, status, and (optionally) raw request/response
bodies. It is the cheapest way to observe what your stack is doing.

Plumbing detail: `on_event` is threaded into `TelemetryModule` (and
into `CircuitBreakerModule` for state-change events). It fires **outside
any locks** so it cannot stall the call path.

For deeper coverage of the telemetry surface — phase timings, cost
calculation, agent attribution — see `09-telemetry-module.ipynb`.

In [15]:
events: list = []


def capture(record):
    events.append(record)


# Mock the inner adapter to avoid network. We patch _build_adapter so
# every internal load returns our mock.
clear_cache()

mock_adapter = make_mock_inner()


def fake_build_adapter(*_args, **_kwargs):
    return mock_adapter


with patch("arcllm.registry._build_adapter", side_effect=fake_build_adapter):
    model = load_model(
        "anthropic",
        telemetry=True,  # required for on_event to wire in
        retry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        on_event=capture,
    )
    for _ in range(3):
        await model.invoke([Message(role="user", content="hi")])

print(f"captured {len(events)} events")
for ev in events:
    print(
        f"  {ev.provider}/{ev.model} status={ev.status} tokens={ev.input_tokens}+{ev.output_tokens}"
    )

captured 3 events
  anthropic/claude-sonnet-4-6 status=success tokens=12+4
  anthropic/claude-sonnet-4-6 status=success tokens=12+4
  anthropic/claude-sonnet-4-6 status=success tokens=12+4


Each entry is a `TraceRecord` (a frozen pydantic model). Inspect one:

In [16]:
rec = events[0]
print("type:", type(rec).__name__)
print("provider:", rec.provider)
print("model:", rec.model)
print("status:", rec.status)
print("phase_timings keys:", list(rec.phase_timings.keys()))
print("input_tokens:", rec.input_tokens, " output_tokens:", rec.output_tokens)
print("agent_label:", rec.agent_label)

type: TraceRecord
provider: anthropic
model: claude-sonnet-4-6
status: success
phase_timings keys: ['prompt_assembly_ms', 'llm_call_ms', 'post_processing_ms', 'total_ms']
input_tokens: 12  output_tokens: 4
agent_label: None


The callback is **synchronous**: don't do blocking I/O in it. If you
need to ship records to a database, queue them and have a background
worker drain the queue. For persistent storage, prefer `trace_store=` —
covered next.

## 6. `trace_store` — persistent, hash-chained recording

`trace_store` accepts any object implementing the `TraceStore` Protocol
(see `arcllm/trace_store.py`). The bundled implementation,
`JSONLTraceStore`, writes records to daily-rotated JSONL files with a
SHA-256 hash chain over the contents. Tampering with a previously
written record breaks the chain — `verify_chain()` will report it.

Threading rules:

- `on_event` and `trace_store` are **independent**: pass either, both,
  or neither.
- The trace store sees only `success` and `error` records — circuit
  breaker `circuit_change` events go through `on_event` only.
- File permissions are tightened to `0o600` after every write
  (NIST AU-9). Directory is created `0o700`.

Deep coverage — query, pagination, tamper detection — lives in
`14-trace-store.ipynb`. Here we just thread the store through
`load_model`.

In [17]:
import tempfile
from arcllm.trace_store import JSONLTraceStore

tmpdir = tempfile.mkdtemp(prefix="arc-traces-")
store = JSONLTraceStore(Path(tmpdir))
print("trace store rooted at:", tmpdir)

trace store rooted at: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc-traces-lama8jmo


In [18]:
clear_cache()

with patch("arcllm.registry._build_adapter", side_effect=fake_build_adapter):
    traced = load_model(
        "anthropic",
        telemetry=True,
        retry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        trace_store=store,
        agent_label="agent-007",
    )
    for _ in range(2):
        await traced.invoke([Message(role="user", content="hello")])

# Verify the chain on disk
ok = await store.verify_chain()
print("chain verified:", ok)

# Read records back
results, _cursor = await store.query(limit=10)
print("records on disk:", len(results))
for r in results:
    print(f"  {r.agent_label} | {r.provider}/{r.model} | status={r.status}")

chain verified: True
records on disk: 2
  agent-007 | anthropic/claude-sonnet-4-6 | status=success
  agent-007 | anthropic/claude-sonnet-4-6 | status=success


See `14-trace-store.ipynb` for record schema, daily rotation,
pagination, and the tamper-detection demo.

## 7. `agent_label` — multi-agent attribution

When multiple agents share a process (or a trace store), `agent_label`
tags every emitted record so you can filter and aggregate per-agent.
The label is a short opaque string — no validation beyond what
`TraceRecord` demands. It surfaces in:

- `TraceRecord.agent_label` on every record (and therefore in `on_event`
  callbacks and the `trace_store`)
- The `agent` query filter on `JSONLTraceStore.query(...)`
- OTel span attributes when `OtelModule` is enabled

In [19]:
clear_cache()
events_a, events_b = [], []

with patch("arcllm.registry._build_adapter", side_effect=fake_build_adapter):
    agent_a = load_model(
        "anthropic",
        telemetry=True,
        retry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        on_event=events_a.append,
        agent_label="planner",
    )
    agent_b = load_model(
        "anthropic",
        telemetry=True,
        retry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        on_event=events_b.append,
        agent_label="executor",
    )

    await agent_a.invoke([Message(role="user", content="plan something")])
    await agent_b.invoke([Message(role="user", content="execute step 1")])
    await agent_b.invoke([Message(role="user", content="execute step 2")])

print("planner events:", [e.agent_label for e in events_a])
print("executor events:", [e.agent_label for e in events_b])

planner events: ['planner']
executor events: ['executor', 'executor']


Two agents, two labels, no cross-pollution. Pair this with a shared
`trace_store=` and you can ship per-agent dashboards from one JSONL
file.

## 8. `routing` — classification-based dispatch

When `routing=` is set, `load_model()` replaces the single adapter at
the innermost stack position with a `RoutingModule`. The router holds
one adapter per *classification* and chooses which to call based on a
`classification` kwarg you pass to `invoke()`.

Use cases:

- Send unclassified data to a cheap commercial provider; route CUI to a
  Gov-cloud provider.
- Route long-context queries to one model, short-fast ones to another.
- A/B test providers per classification.

For a full deep-dive (NFKC validation, enforcement modes,
classification taxonomy), see `arcllm/17-routing-module.ipynb`. Here we
focus on the `load_model()` integration.

In [20]:
clear_cache()


def fake_build_router_adapter(provider_name, *_args, **_kwargs):
    """Return a distinct mock per provider so we can tell which was called."""
    return make_mock_inner(name=provider_name, model=f"{provider_name}-test")


with patch("arcllm.registry._build_adapter", side_effect=fake_build_router_adapter):
    router_model = load_model(
        "anthropic",  # primary provider — overridden by routing rules
        telemetry=False,
        retry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        routing={
            "rules": {
                "unclassified": {"provider": "openai"},
                "cui": {"provider": "anthropic"},
            },
            "default_classification": "unclassified",
            "enforcement": "block",
        },
    )

print("router stack:", " -> ".join(stack_layers(router_model)))

router stack: RoutingModule


In [21]:
# Dispatch by classification kwarg.
r1 = await router_model.invoke(
    [Message(role="user", content="public data")],
    classification="unclassified",
)
r2 = await router_model.invoke(
    [Message(role="user", content="sensitive data")],
    classification="cui",
)
print("unclassified -> model:", r1.model)
print("cui -> model:", r2.model)

unclassified -> model: openai-test
cui -> model: anthropic-test


Calls without a `classification` kwarg fall back to
`default_classification`. With `enforcement="block"`, an unknown
classification raises; with `enforcement="warn"`, it logs and falls
back. See `17-routing-module.ipynb` for the full contract.

## 9. `circuit_breaker` — fail-fast on unhealthy providers

`CircuitBreakerModule` is a per-provider state machine:

- **CLOSED** — normal operation; failures increment a counter.
- **OPEN** — after `failure_threshold` consecutive failures, all calls
  raise `CircuitOpenError` immediately for `cooldown_seconds`.
- **HALF_OPEN** — after cooldown, allow up to `half_open_max_calls`
  probes. One success → CLOSED; one failure → OPEN again.

State changes emit `circuit_change` `TraceRecord`s through `on_event`.
This is how UIs surface red/yellow/green health in real time. Full
deep-dive in `15-queue-circuit-breaker.ipynb`.

In [22]:
from arcllm.modules.circuit_breaker import CircuitOpenError, CircuitState
from arcllm.exceptions import ArcLLMAPIError

clear_cache()


def always_503(*_args, **_kwargs):
    raise ArcLLMAPIError(503, "upstream 503", "anthropic")


broken_inner = MagicMock(spec=LLMProvider)
broken_inner.name = "anthropic"
broken_inner.model_name = "claude-sonnet-4-6"
broken_inner.invoke = AsyncMock(side_effect=always_503)

state_changes: list = []

with patch("arcllm.registry._build_adapter", return_value=broken_inner):
    cb_model = load_model(
        "anthropic",
        telemetry=False,
        retry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        otel=False,
        circuit_breaker={"failure_threshold": 3, "cooldown_seconds": 0.5},
        on_event=state_changes.append,
    )

# Trip it: drive 3 failures, then verify the 4th is rejected without ever
# touching the inner.
for i in range(3):
    try:
        await cb_model.invoke([Message(role="user", content="x")])
    except ArcLLMAPIError:
        pass

print("inner calls so far:", broken_inner.invoke.await_count)

try:
    await cb_model.invoke([Message(role="user", content="x")])
except CircuitOpenError as e:
    print("rejected (no inner call):", e)

print("inner calls after circuit open:", broken_inner.invoke.await_count)
print("state-change events captured:", len(state_changes))
for ev in state_changes:
    print(f"  {ev.event_type}: {ev.event_data}")

inner calls so far: 3
rejected (no inner call): Circuit open for 'anthropic': 0.5s remaining in cooldown
inner calls after circuit open: 3
state-change events captured: 1
  circuit_change: {'provider': 'anthropic', 'old_state': 'closed', 'new_state': 'open', 'consecutive_failures': 3}


## 10. `queue` — bounded concurrency with backpressure

`QueueModule` gates `invoke()` through an `asyncio.BoundedSemaphore`. At
most `max_concurrent` calls hit the inner provider; excess callers wait
in FIFO. If the wait queue itself exceeds `max_queued`, new calls are
**rejected** with `QueueFullError` rather than queued forever.

Why this matters: providers have hard rate limits and connection caps.
Without a queue, a burst from a coordinator (e.g. 50 parallel agent
turns) overshoots the rate limit and cascades failures. The queue
absorbs the burst.

Full coverage in `15-queue-circuit-breaker.ipynb`.

In [23]:
import asyncio

clear_cache()


async def slow_invoke(*_args, **_kwargs):
    await asyncio.sleep(0.05)
    return LLMResponse(
        content="ok",
        usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    )


slow_inner = MagicMock(spec=LLMProvider)
slow_inner.name = "anthropic"
slow_inner.model_name = "claude-sonnet-4-6"
slow_inner.invoke = AsyncMock(side_effect=slow_invoke)

with patch("arcllm.registry._build_adapter", return_value=slow_inner):
    q_model = load_model(
        "anthropic",
        telemetry=False,
        retry=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        queue={"max_concurrent": 2, "max_queued": 1, "call_timeout": 5.0},
    )

# Three concurrent calls: 2 run, 1 queues, all succeed.
results = await asyncio.gather(
    q_model.invoke([Message(role="user", content="a")]),
    q_model.invoke([Message(role="user", content="b")]),
    q_model.invoke([Message(role="user", content="c")]),
)
print("queued OK:", [r.content for r in results])

queued OK: ['ok', 'ok', 'ok']


In [24]:
# Now overflow the queue: 5 concurrent against capacity 2 + max_queued 1.
results_or_errors = await asyncio.gather(
    *(q_model.invoke([Message(role="user", content=str(i))]) for i in range(5)),
    return_exceptions=True,
)
ok = [r for r in results_or_errors if isinstance(r, LLMResponse)]
rejected = [r for r in results_or_errors if isinstance(r, QueueFullError)]
print("succeeded:", len(ok), "| rejected:", len(rejected))
if rejected:
    print("first rejection:", rejected[0])

Queue backpressure: 1 waiters (max 1)


Queue backpressure: 1 waiters (max 1)


succeeded: 3 | rejected: 2
first rejection: Queue full: 1 calls waiting (max 1)


`QueueFullError` is the signal to retry, switch providers, or fail
the request — never to silently drop work.

## 11. The rest of the kwargs in one section

The remaining seven module kwargs follow the same `True | False | dict |
None` shape. Cross-references point to the dedicated deep-dives.

| Kwarg            | Module                | Job                                                           | Deep-dive notebook        |
|------------------|-----------------------|----------------------------------------------------------------|---------------------------|
| `retry`          | `RetryModule`         | Retry transient errors with exponential backoff                | (covered in 07)           |
| `fallback`       | `FallbackModule`      | On error, try a chain of secondary providers                   | (covered in 07)           |
| `rate_limit`     | `RateLimitModule`     | Token-bucket rate limiting per provider                        | `08-rate-limiter.ipynb`   |
| `audit`          | `AuditModule`         | Emit structured audit logs for compliance                      | `10-audit-trail.ipynb`    |
| `security`       | `SecurityModule`      | PII redaction, request signing                                 | `12-security-layer.ipynb` |
| `telemetry`      | `TelemetryModule`     | Tokens / cost / phase timings; `on_event` and `trace_store`    | `09-telemetry-module.ipynb` |
| `otel`           | `OtelModule`          | OpenTelemetry spans for distributed tracing                    | `11-otel-export.ipynb`    |

Two examples below show the dict-config shape for retry and rate-limit;
the others follow identically.

In [25]:
clear_cache()

# Retry: up to 4 retries, exponential backoff with a 1ms base.
retry_attempts = {"n": 0}


def flaky(*_args, **_kwargs):
    retry_attempts["n"] += 1
    if retry_attempts["n"] < 3:
        raise ArcLLMAPIError(500, "flap", "anthropic")
    return LLMResponse(
        content="recovered",
        usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    )


flap_inner = MagicMock(spec=LLMProvider)
flap_inner.name = "anthropic"
flap_inner.model_name = "claude-sonnet-4-6"
flap_inner.invoke = AsyncMock(side_effect=flaky)

with patch("arcllm.registry._build_adapter", return_value=flap_inner):
    retry_model = load_model(
        "anthropic",
        telemetry=False,
        queue=False,
        security=False,
        audit=False,
        rate_limit=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        retry={"max_retries": 4, "backoff_base_seconds": 0.001},
    )

resp = await retry_model.invoke([Message(role="user", content="hi")])
print(f"retry succeeded after {retry_attempts['n']} attempts:", resp.content)

Retry attempt 1/4 after 0.00s: anthropic API error (HTTP 500): flap


Retry attempt 2/4 after 0.00s: anthropic API error (HTTP 500): flap


retry succeeded after 3 attempts: recovered


In [26]:
clear_cache()

with patch("arcllm.registry._build_adapter", side_effect=fake_build_adapter):
    rl_model = load_model(
        "anthropic",
        telemetry=False,
        queue=False,
        security=False,
        audit=False,
        retry=False,
        fallback=False,
        circuit_breaker=False,
        otel=False,
        rate_limit={"requests_per_minute": 60, "burst_capacity": 5},
    )

print("rate-limited stack:", " -> ".join(stack_layers(rl_model)))
print("RateLimitModule sits *innermost* — closest to the adapter.")

rate-limited stack: RateLimitModule -> MagicMock
RateLimitModule sits *innermost* — closest to the adapter.


## 12. Module stack order — why this order

The canonical stack is:

```
Otel
  └─ Queue
       └─ Telemetry
            └─ Audit
                 └─ Guardrails
                      └─ Injection
                           └─ Security
                                └─ CircuitBreaker
                                     └─ Retry
                                          └─ Fallback
                                               └─ RateLimit
                                                    └─ [Router | LoadBalancer | Adapter]
```

Read from the **outside in** — that's the order kwargs arrive on the
inbound side; responses unwind in the reverse order on the way back out.
This is `registry.load_model`'s documented order (ADR-430) — **do not
reorder** without re-reading that ADR.

The order is not arbitrary. Key constraints, from the source:

- **Otel** (outermost) wraps the entire wall-clock cost of a call so
  span timings include queue wait, retries, the full thing.
- **Queue** is just inside Otel so OTel observes wait time.
- **Telemetry** sits inside Queue and measures the inner-call timing —
  provider latency plus everything below it, but not queue wait.
- **Audit → Guardrails → Injection → Security** is one contiguous block,
  in that order, on purpose:
  - **Injection sits above (outside) Security** so it scans the
    *original* inbound text for prompt-injection patterns before PII/secret
    redaction can obscure the attack signal (OWASP LLM01, ASI06).
  - **Guardrails sits just inside Audit** so it validates the final,
    fully-resolved response (post-Retry/Fallback) and the audit trail
    records its verdict.
- **CircuitBreaker** sits *below* Security — a tripped breaker still lets
  audit/guardrails/injection/security see and log the call, but rejects
  before ever reaching Retry/Fallback/RateLimit/the adapter.
- **Retry** wraps **Fallback** so a single retry attempt can fall back
  through providers, then retry again.
- **Fallback** wraps **RateLimit** so each fallback target has its own
  rate-limit budget.
- **RateLimit** sits innermost — the last gate before the adapter — so
  every successful call counts against the bucket once, regardless of
  retries from outer layers.

The `_inner` chain on a default `load_model("anthropic")` reflects this
order exactly (skipping whichever layers aren't enabled in
`config.toml`). Let's print it.

In [27]:
clear_cache()
default = load_model("anthropic")
print("default stack:", " -> ".join(stack_layers(default)))

default stack: QueueModule -> TelemetryModule -> AuditModule -> RetryModule -> FallbackModule -> RateLimitModule -> AnthropicAdapter


When debugging, walk the chain to see which layers are active.
Modules disabled in `config.toml` simply don't appear.

In [28]:
def describe_stack(provider) -> None:
    """Pretty-print every layer with its config."""
    cur = provider
    depth = 0
    while True:
        cls = type(cur).__name__
        cfg = getattr(cur, "_config", None)
        cfg_keys = sorted(cfg.keys()) if isinstance(cfg, dict) else []
        prefix = "  " * depth
        print(f"{prefix}{cls}  config_keys={cfg_keys}")
        inner = getattr(cur, "_inner", None)
        if inner is None:
            break
        cur = inner
        depth += 1


describe_stack(default)

QueueModule  config_keys=['call_timeout', 'max_concurrent', 'max_queued']
  TelemetryModule  config_keys=['classification', 'cost_cache_read_per_1m', 'cost_cache_write_per_1m', 'cost_input_per_1m', 'cost_output_per_1m', 'default_max_tokens', 'encryption', 'log_level', 'store_raw_bodies']
    AuditModule  config_keys=[]
      RetryModule  config_keys=['backoff_base_seconds', 'max_retries']
        FallbackModule  config_keys=['chain']
          RateLimitModule  config_keys=['burst_capacity', 'requests_per_minute']
            AnthropicAdapter  config_keys=[]


## 13. Multi-tool loop end-to-end (mock, all bells on)

A complete agentic loop with tools, telemetry, audit, retry, and a
trace store. The mock adapter scripts a deterministic two-turn
conversation: turn 1 returns `tool_use`, turn 2 returns the final
text answer.

In [29]:
calculator_tool = Tool(
    name="calculate",
    description="Evaluate a math expression.",
    parameters={
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
)


def execute_calculate(args: dict) -> str:
    expr = args["expression"]
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expr):
        return f"unsafe expression: {expr!r}"
    try:
        return str(eval(expr))
    except Exception as e:
        return f"error: {e}"


print("tool ready:", calculator_tool.name)

tool ready: calculate


In [30]:
# Scripted two-turn response from the mock adapter.
SCRIPTED = [
    LLMResponse(
        content="I'll compute that.",
        tool_calls=[
            ToolCall(id="call-1", name="calculate", arguments={"expression": "137 * 42 + 19"})
        ],
        usage=Usage(input_tokens=20, output_tokens=15, total_tokens=35),
        model="claude-sonnet-4-6",
        stop_reason="tool_use",
    ),
    LLMResponse(
        content="The answer is 5773.",
        tool_calls=[],
        usage=Usage(input_tokens=40, output_tokens=10, total_tokens=50),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    ),
]

scripted_inner = MagicMock(spec=LLMProvider)
scripted_inner.name = "anthropic"
scripted_inner.model_name = "claude-sonnet-4-6"
scripted_inner.invoke = AsyncMock(side_effect=SCRIPTED)
print("scripted inner ready (2 turns)")

scripted inner ready (2 turns)


In [31]:
clear_cache()

events_full: list = []
tmp2 = tempfile.mkdtemp(prefix="arc-traces-full-")
store2 = JSONLTraceStore(Path(tmp2))

with patch("arcllm.registry._build_adapter", return_value=scripted_inner):
    full = load_model(
        "anthropic",
        telemetry=True,
        retry={"max_retries": 2, "backoff_base_seconds": 0.001},
        queue={"max_concurrent": 4},
        audit=True,
        security=False,  # keep messages unmodified for the demo
        rate_limit=False,
        fallback=False,
        circuit_breaker={"failure_threshold": 5, "cooldown_seconds": 30},
        otel=False,
        on_event=events_full.append,
        trace_store=store2,
        agent_label="multi-tool-demo",
    )

print("full stack:", " -> ".join(stack_layers(full)))

full stack: QueueModule -> TelemetryModule -> AuditModule -> CircuitBreakerModule -> RetryModule -> MagicMock


In [32]:
# Turn 1: send the question with the tool available.
messages = [Message(role="user", content="What is 137 * 42 + 19?")]
turn1 = await full.invoke(messages, tools=[calculator_tool])
print("turn 1 stop_reason:", turn1.stop_reason)
print("turn 1 tool_calls:", [(tc.name, tc.arguments) for tc in turn1.tool_calls])

turn 1 stop_reason: tool_use
turn 1 tool_calls: [('calculate', {'expression': '137 * 42 + 19'})]


In [33]:
# Execute the tool call from turn 1.
tc = turn1.tool_calls[0]
result_str = execute_calculate(tc.arguments)
print("local execute_calculate:", result_str)

# Build assistant message with the tool_use block, then the tool_result message.
asst_blocks = []
if turn1.content:
    asst_blocks.append(TextBlock(text=turn1.content))
asst_blocks.append(ToolUseBlock(id=tc.id, name=tc.name, arguments=tc.arguments))

messages.append(Message(role="assistant", content=asst_blocks))
messages.append(
    Message(role="tool", content=[ToolResultBlock(tool_use_id=tc.id, content=result_str)])
)

# Turn 2: send the tool result back, get the final answer.
turn2 = await full.invoke(messages, tools=[calculator_tool])
print("turn 2 stop_reason:", turn2.stop_reason)
print("turn 2 content:", turn2.content)

local execute_calculate: 5773
turn 2 stop_reason: end_turn
turn 2 content: The answer is 5773.


In [34]:
# Verify the trace store and on_event captured everything.
print("on_event records:", len(events_full))
ok = await store2.verify_chain()
print("trace store chain verified:", ok)
results, _cur = await store2.query(agent="multi-tool-demo", limit=10)
print("records on disk:", len(results))
for r in results:
    print(f"  {r.event_type} | tokens {r.input_tokens}+{r.output_tokens} | status {r.status}")

on_event records: 2
trace store chain verified: True
records on disk: 2
  llm_call | tokens 40+10 | status success
  llm_call | tokens 20+15 | status success


## 14. (live) one real call through `load_model`

Gated by `ANTHROPIC_API_KEY`. Skips if the key isn't set. This exercises
the *full* default stack against the real Anthropic API — useful as a
smoke test that your config + credentials are working.

In [35]:
async def live_smoke_test() -> None:
    clear_cache()

    # A small live call. Default stack: telemetry, queue, audit, security,
    # retry — whatever your arcllm.toml has enabled.
    live_events: list = []
    live = load_model(
        "anthropic",
        "claude-haiku-4-5-20251001",
        on_event=live_events.append,
        agent_label="live-smoke",
    )
    resp = await live.invoke(
        [Message(role="user", content="Reply with the single word: pong.")],
        max_tokens=8,
    )
    print("live response:", resp.content)
    print("usage:", resp.usage)
    print("captured events:", len(live_events))
    await live.close()


if HAS_LIVE_KEY:
    await live_smoke_test()
else:
    print("skip — set ANTHROPIC_API_KEY to run this section")

skip — set ANTHROPIC_API_KEY to run this section


## 15. Summary

`load_model()` is the only function you need to call to construct a
production-ready LLM client in arcllm. Two arguments give you a default
stack; thirteen kwargs let you tune everything.

What we covered:

- **Adapter contract** — `LLMProvider.invoke(messages, tools, **kwargs)`
  is what every layer wraps. Adapters are async-context-manager safe.
- **Module pattern** — `BaseModule(config, inner)` is a `_inner`-chained
  decorator. You can stack modules manually, but `load_model()` does it
  right for you.
- **Every kwarg of `load_model()`**: `provider`, `model`, `on_event`,
  `trace_store`, `agent_label`, `routing`, `circuit_breaker`, `queue`,
  `retry`, `fallback`, `rate_limit`, `audit`, `security`, `telemetry`,
  `otel` (plus `budget_scope`).
- **Three-state semantics** — every module kwarg accepts `None` (use
  config.toml), `True`/`False` (force on/off), or a `dict` (force on
  with overrides merged over config.toml).
- **The canonical stack order** —
  `Otel → Queue → Telemetry → CircuitBreaker → Audit → Security →
  Retry → Fallback → RateLimit → [Router|Adapter]` — and *why* each
  layer sits where it does.
- **Walking the `_inner` chain** for live debugging: every `BaseModule`
  exposes `_inner`; the adapter is the bottom.
- **Mock-first patterns**: `patch("arcllm.registry._build_adapter")` is
  the cleanest way to swap in a fake provider for any `load_model()`
  call.

Where to go next:

- `09-telemetry-module.ipynb` — `TelemetryModule` deep dive: phase
  timings, cost calculation, budget enforcement.
- `14-trace-store.ipynb` — `JSONLTraceStore`: hash chains, daily
  rotation, tamper detection, query API.
- `15-queue-circuit-breaker.ipynb` — `QueueModule` and
  `CircuitBreakerModule` end-to-end with concurrency tests.
- `17-routing-module.ipynb` — `RoutingModule`: classification taxonomy,
  NFKC validation, enforcement modes.
- `06-provider-registry.ipynb` — registry internals: provider name
  validation, caching, multi-provider routing.
- `07-module-system.ipynb` — `BaseModule` contract, custom modules,
  hook lifecycle.

Public API exercised in this notebook:

`arcllm.load_model`, `arcllm.clear_cache`, `arcllm.AnthropicAdapter`,
`arcllm.AuditModule`, `arcllm.BaseModule`, `arcllm.OtelModule`,
`arcllm.QueueModule`, `arcllm.RateLimitModule`, `arcllm.RetryModule`,
`arcllm.SecurityModule`, `arcllm.TelemetryModule`,
`arcllm.modules.routing.RoutingModule`,
`arcllm.modules.circuit_breaker.CircuitBreakerModule`
(and `CircuitOpenError`, `CircuitState`),
`arcllm.exceptions.QueueFullError`, `arcllm.exceptions.ArcLLMAPIError`,
`arcllm.exceptions.ArcLLMConfigError`, `arcllm.types.LLMProvider`,
`arcllm.trace_store.JSONLTraceStore`, `arcllm.trace_store.TraceRecord`,
plus the data types `Message`, `Tool`, `ToolCall`, `ToolUseBlock`,
`ToolResultBlock`, `TextBlock`, `LLMResponse`, `Usage`.